In [3]:
import utils
import json
import pandas as pd
from dotenv import load_dotenv


In [4]:
# make it easy to call LLMs hosted by different like OAI/Claude/Grok...
import aisuite as ai


In [5]:
client = ai.Client()

In [8]:
# here We use SQLite(relational Database) smaller dat interaction postsql
utils.print_html(utils.get_schema('products.db'))

In [25]:
#now this generate SQL generator

def generate_sql(questions: str, schema:str, model:str) -> str:
    prompt= f"""
    You are SQL assistant, return the schema and the user's questions about relational DB

    Schema:
    {schema}

    User Questions:
    {questions}
    Respond with SQL(relational DB) only
    """
    response = client.chat.completions.create(
        model = model,
        messages = [
            {"role": "user",
             "content": prompt}
        ],
        temperature = 0,
    )
    return response.choices[0].message.content.strip()

In [26]:
# now running above function
schema = """
Table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""
question = "Which color of product has the highest total sales?"
utils.print_html(question, title = "User Question")
first_try_sql = generate_sql(question, schema, model="openai:gpt-4.1")

# __formatted__ result
utils.print_html(first_try_sql, title = "SQL query first try")


In [28]:
# now we got the SQL query based on the USer asking 
# 이제 내가 한대로 라지모델이 쿼리를 줬으니 이걸를 실해시켜 보는단계

df_sql_first_try = utils.execute_sql(first_try_sql, db_path = 'products.db')

In [29]:
utils.print_html(df_sql_first_try, title = "output of sql first try is shit")

color,total_sales
blue,-190571.46


In [42]:
# now the answer is incorrect so we need reflection stage. 
# so FAR we did just input gave output so now we need to reflect 
# ask LLM to review SQL query with original question and schema 
def refine_sql(
    question:str,
    sql_query:str,
    schema: str,
    model: str) -> tuple[str,str]:
    prompt = f"""
    You are a SQL reviewer and refiner.
    
    User asked:
    {question}
    
    Original SQL:
    {sql_query}
    
    Table Schema:
    {schema}
    
    Step 1: Briefly evaluate if the SQL OUTPUT fully answers the user's question.
    Step 2: If improvement is needed, provide a refined SQL query for SQLite.
    If the original SQL is already correct, return it unchanged.
    
    Return STRICT JSON with two fields:
    {{
      "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
      "refined_sql": "<final SQL to run>"
    }}
    """
    response = client.chat.completions.create(
        model = model,
        messages = [
            {
                "role": "user", 
                "content": prompt}],
        temperature = 0,
        )
    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback =str(obj.get("feedback", "")).strip()
        second_try_sql = str(obj.get("second_try_sql", sql_query)).strip()
        if not second_try_sql:
            second_try_sql = sql_query
    except Exception:
        feedback = content.strip()
        second_try_sql = sql_query

    return feedback, second_try_sql
    
        

In [44]:
#refractor generated SQL from first try to second try 
feedback, second_try_sql = refine_sql(
    question = question,
    sql_query = first_try_sql,
    schema = schema,
    model = "openai:gpt-4.1"
)

#show origial version sql (incorrect try)
utils.print_html(question, title = "User Question")

# this is first try  SQL 
utils.print_html(first_try_sql, title = "First generated SQL query")

df_sql_v1=utils.execute_sql(first_try_sql, db_path = "products.db")
utils.print_html(df_sql_v1, title = "sql out of Version1")

# refined and 2nd try
utils.print_html(feedback, title ="feedback of v1")
utils.print_html(second_try_sql, title= "refractored sql query")

df_sql_v2 = utils.execute_sql(second_try_sql, db_path = "products.db")
utils.print_html(df_sql_v2, title = "sql version2")

color,total_sales
blue,-190571.46


color,total_sales
blue,-190571.46


In [47]:
# from refractor above,still showing negative output because its only checkin Queries instead
# of actual data

def refine_sql_external_feedback(
    question:str,
    sql_query:str,
    df_feedback: pd.DataFrame,
    schema:str,
    model: str)->tuple[str,str]:
    prompt = f"""
    You are a SQL reviewer and refiner.

    User asked:
    {question}

    Original SQL:
    {sql_query}

    SQL Output:
    {df_feedback.to_markdown(index=False)}

    Table Schema:
    {schema}

    Step 1: Briefly evaluate if the SQL output answers the user's question.
    Step 2: If the SQL could be improved, provide a refined SQL query.
    If the original SQL is already correct, return it unchanged.

    Return a strict JSON object with two fields:
    - "feedback": brief evaluation and suggestions
    - "refined_sql": the final SQL to run
    """

    response = client.chat.completions.create(
        model = model,
        messages = [{"role":"user", "content": prompt}],
        temperature =1.0,
    )
    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_qeury
    except Exception:
        # model doesnt run JSON
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql 
    

In [59]:
# disply first generated sql query version1
# output of V1 
# check the answer of V1 + give feedback on v1
# refine query and create new SQL v2 
# run V2 and display results 

# original version of v1
df_sql_v1 = utils.execute_sql(first_try_sql, db_path = "products.db")

# use external feedback and evaluate 
feedback, second_try_sql = refine_sql_external_feedback(
    question= question,
    sql_query= first_try_sql,
    df_feedback = df_sql_v1,
    schema = schema,
    model = "openai:gpt-4.1"
)

# V1 
utils.print_html(question, title = "this is user question")
utils.print_html(first_try_sql, title = "generated sql") 
utils.print_html(df_sql_v1, title = "output of sql v1") 

# feedback and new SQL V2 
utils.print_html(feedback, title = "feedback on v1") 
utils.print_html(second_try_sql, title = "refined sql query version2")

# execute and display v2 results
df_sql_v2 = utils.execute_sql(second_try_sql, db_path = "products.db") 
utils.print_html(df_sql_v2, title = "output of sql v2")

color,total_sales
blue,-190571.46


color,total_sales
white,358315.09


In [64]:
# now all together
# 1.extract database schema
# 2.generate initial SQL 
# 3.reflect on V1 using execution feedback
# 4. refined feedback V2 query

def agentic_sql_workflows(
    db_path: str,
    question:str,
    model_generation:str = "openai:gpt-4.1",
    model_evaluation: str= "openai:gpt-4.1"):

    #getting schema
    schema = utils.get_schema(db_path)
    utils.print_html(
        schema,
        title = "generate SQL v1")

    # run generated sql 
    df_v1 = utils.execute_sql(first_try_sql, db_path)
    utils.print_html(
        df_v1,
        title = "running 1st generated sql")

    # reflect on output of V1 and refractor to V2
    feedback, second_try_sql = refine_sql_external_feedback(
        question = question,
        sql_query = first_try_sql,
        df_feedback= df_v1,
        schema = schema,
        model = model_evaluation,)

    utils.print_html(
        feedback,
        title = "refelect on v1")
    utils.print_html(
        second_try_sql,
        title = "refractored sql v2")
    
    df_v2 = utils.execute_sql(second_try_sql, db_path)
    utils.print_html(df_v2, title = "executing v2")


In [65]:
agentic_sql_workflows(
    "products.db", 
    "Which color of product has the highest total sales?",
    model_generation="openai:gpt-4.1",
    model_evaluation="openai:gpt-4.1"
)

color,total_sales
blue,-190571.46


color,total_sales
white,358315.09
